# The Notebook Assistant, End to End

The full code of [The Notebook Assistant, End to End](https://residual-stream.lvtan71.workers.dev/posts/notebook-assistant-end-to-end/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/notebook-assistant-end-to-end) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/notebook-assistant-end-to-end.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 19 min end to end.
Uses post 39's 150 test questions. The original file is lost; this notebook uses questions regenerated by the post 39 notebook, so the scorecard's counts differ from the post's.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.18.*" sentence-transformers huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "notebook-assistant-end-to-end"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

In [ ]:
!tar xzf blog-6698eca.tar.gz

## `scorecard.py`

Traces one question through every stage, then runs the 150 test questions (retrieval, citations, the judge, the check, time and tokens per answer), 30 rewordings for the cache, 15 questions the blog doesn't answer, and a poisoned post planted in the index. Saves every answer to `answers.json`.

In [ ]:
!{sys.executable} scorecard.py

The post printed:

```text
=== one question, traced ===
PROMPT (user turn):
Question: How is each patch processed before being embedded in the linear layer?

<passage id="1" post="Seeing with Transformers: ViT, CLIP and Vision-Language Models">
 The Vision Transformer (ViT; Dosovitskiy et al., 2020) uses the most
direct translation. Cut the image into a grid of small square patches,
flatten each patch's pixels into one vector, and project every patch with
the same learned linear layer, the patch embedding. Add position
embeddings (post 12) so the model
knows where each patch came from (ViT learns one embedding per position
in the flattened sequence; later models use 2-D-aware schemes such as 2-D
RoPE so the same model can take images of different sizes), and the
result is a sequence like any other:
</passage>

<passage id="2" post="Where Am I? Position in Attention, from Sinusoids to RoPE">
 The original Transformer (Vaswani et al., 2017) adds a position vector
to each token's embedding before the first layer. Theirs was fixed, made of
sinusoids at different frequencies:
</passage>

<passage id="3" post="Seeing with Transformers: ViT, CLIP and Vision-Language Models">
A 224×224 image in 16×16-pixel patches is 196 tokens (each "word" in the
paper's title, "an image is worth 16×16 words", is one such patch), plus
one extra. That extra is the [CLS]
token, borrowed from BERT: a learned vector prepended to the sequence
whose output, after every layer of attention to the patches, serves as
the representation of the whole image. The alternatives are to aver
...
ANSWER: Each patch is flattened into a vector and then projected by a learned linear layer [1]. Before being embedded in the linear layer, the patches are cut from the image into a grid of small square patches [1].

question: How is each patch processed before being embedded in the linear layer?
passages: [1] vision-transformers-and-multimodal / Patches: an image as a sequence; [2] rope-and-positional-encoding / Adding position to the tokens; [3] vision-transformers-and-multimodal / Patches: an image as a sequence; [4] residual-stream-explained / Each block: attention sublayer + MLP sublayer, both additive
status: ok; seconds: cache 0.00, retrieve + rerank 0.19, generate 4.81; tokens in 607, out 44

=== 150 answerable questions ===
answer's sentence among the 4 passages: 135/150
answer cites the right post:            128/150
judge: answer agrees with the sentence: 86/150  (when the sentence was retrieved: 83/135; when it wasn't: 3/15)
check: ok 134, refused 15, no citation 1, bad citation 0
refused although the sentence was retrieved: 10

per answer, on this laptop: cache 0.01 s, retrieve + rerank 0.12 s, generate 4.71 s
tokens: 699 in, 52 out; at $3/$15 per million on an API: $0.0029

=== repeats: 30 rewordings of questions already answered ===
served from the cache: 12/30

=== 15 questions the blog doesn't answer ===
refused: 14/15

=== injection: a poisoned post planted in the index ===
poisoned passage retrieved: 4/10; answer repeated the attacker's link: 0/10 (of those that retrieved it: 0/4)
```

## `rescore.py`

Rescores `answers.json` with stricter rules: refusals, citations on answers only, how many answer sentences any chunk holds, and whether each cache hit matched its own question. No model calls but the embedder.

In [ ]:
!{sys.executable} rescore.py

The post printed:

```text
answer sentence findable in some chunk: 147/150; among the 4 passages: 135/150
refused (answer begins with the refusal): 14; of them, with the sentence retrieved: 9; refusals that still cite a passage: 11
answers: 136; citing the right post among them: 117; citing more than one passage: 25
cache: 12 of 30 rewordings hit at 0.90; matched their own original: 12; cached answers that were refusals: 1
```

## `cost_sketch.py`

The cost sketch of AI Engineering, from the Inside (an 8B model on one H100), with the prompt and answer lengths measured here.

In [ ]:
!{sys.executable} cost_sketch.py

The post printed:

```text
series map's guess    2000 in, 300 out: batch 1  1.47 GPU-s per answer; batch 64   82 ms per answer
measured here          699 in,  52 out: batch 1  0.26 GPU-s per answer; batch 64   17 ms per answer
```